In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class MHA(nn.Module):

    def __init__(self, input_dim, num_head):
        super().__init__()
        assert input_dim % num_head == 0
        self.input_dim = input_dim
        self.num_head = num_head
        self.d_k = input_dim // num_head
        self.wqkv = nn.Linear(input_dim, input_dim * 3)
        self.wo = nn.Linear(input_dim, input_dim)

    def forward(self, x, mask=None):
        B, L, D = x.shape
        qkv = self.wqkv(x)
        q, k, v = torch.chunk(qkv, 3, dim=-1)
        q = q.view(B, L, self.num_head, self.d_k).transpose(1, 2)
        k = k.view(B, L, self.num_head, self.d_k).transpose(1, 2)
        v = v.view(B, L, self.num_head, self.d_k).transpose(1, 2)
        attention = torch.matmul(q, k.transpose(-1, -2)) / math.sqrt(self.d_k)
        if mask is not None:
            attention = attention.masked_fill(mask == 0, -1e9)
        attention_scores = F.softmax(attention, dim=-1)
        context = torch.matmul(attention_scores, v)
        context = context.transpose(1, 2).contiguous().view(B, L, D)
        output = self.wo(context)
        return output
B, L, D = 10, 20, 36
x = torch.randn(B, L, D)
mha = MHA(D, 2)

print(mha(x).shape)

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class MQA(nn.Module):

    def __init__(self, input_dim, num_head):
        super().__init__()
        assert input_dim % num_head == 0
        self.input_dim = input_dim
        self.num_head = num_head
        self.d_k = input_dim // num_head
        self.wq = nn.Linear(input_dim, input_dim)
        self.wkv = nn.Linear(input_dim, self.d_k * 2)
        self.wo = nn.Linear(input_dim, input_dim)

    def forward(self, x, mask=None):
        B, L, D = x.shape
        q = self.wq(x)
        kv = self.wkv(x)
        k, v = torch.chunk(kv, 2, dim=-1)
        q = q.view(B, L, self.num_head, self.d_k).transpose(1, 2)
        k = torch.unsqueeze(k, 1)   # (B, 1, L, d_k)
        v = torch.unsqueeze(v, 1)   # (B, 1, L, d_k)
        attention = torch.matmul(q, k.transpose(-1, -2)) / math.sqrt(self.d_k)
        if mask is not None:
            attention = attention.masked_fill(mask == 0, -1e9)
        attention_scores = F.softmax(attention, dim=-1)
        context = torch.matmul(attention_scores, v)
        context = context.transpose(1, 2).contiguous().view(B, L, D)
        output = self.wo(context)
        return output
B, L, D = 10, 20, 36
x = torch.randn(B, L, D)
mha = MQA(D, 2)

print(mha(x).shape)

torch.Size([10, 20, 36])


In [ ]:
class MQA(nn.Module):
    def __init__(self, input_dim, num_head):
        super().__init__()
        self.num_head = num_head
        self.d_k = input_dim // num_head
        self.wq  = nn.Linear(input_dim, input_dim)
        self.wkv = nn.Linear(input_dim, self.d_k * 2)
        self.wo = nn.Linear(input_dim, input_dim)
    
    def forward

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class GQA(nn.Module):

    def __init__(self, input_dim, q_num_head, kv_num_head):
        super().__init__()
        assert input_dim % q_num_head == 0
        assert q_num_head % kv_num_head == 0
        self.input_dim = input_dim
        self.q_num_head = q_num_head
        self.kv_num_head = kv_num_head
        self.d_k = input_dim // q_num_head
        self.num_q_group = q_num_head // kv_num_head

        self.wq = nn.Linear(input_dim, input_dim)
        self.wkv = nn.Linear(input_dim, self.d_k * kv_num_head * 2)
        self.wo = nn.Linear(input_dim, input_dim)

    def forward(self, x, mask=None):
        B, L, D = x.shape
        q = self.wq(x)
        q = q.view(B, L, self.q_num_head, self.d_k).transpose(1, 2).contiguous().view(B, self.kv_num_head, self.num_q_group, L, self.d_k)
        kv = self.wkv(x)
        k, v = torch.chunk(kv, 2, dim=-1)
        k = k.view(B, L, self.kv_num_head, self.d_k).transpose(1, 2)
        v = v.view(B, L, self.kv_num_head, self.d_k).transpose(1, 2)
        k = torch.unsqueeze(k, 2)   # (B, self.kv_num_head, 1, L, d_k)
        v = torch.unsqueeze(v, 2)   # (B, self.kv_num_head, 1, L, d_k)
        attention = torch.matmul(q, k.transpose(-1, -2)) / math.sqrt(self.d_k)
        if mask is not None:
            attention = attention.masked_fill(mask == 0, -1e9)
        attention_scores = F.softmax(attention, dim=-1)
        context = torch.matmul(attention_scores, v)
        context = context.view(B, self.q_num_head, L, self.d_k).transpose(1, 2).contiguous().view(B, L, D)
        output = self.wo(context)
        return output
B, L, D = 10, 20, 36
x = torch.randn(B, L, D)
mha = GQA(D, 4, 2)

print(mha(x).shape)

torch.Size([10, 20, 36])


In [25]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, activation="gelu", norm="LN", dropout=0.1):        
        super().__init__()
        layer_dim = [input_dim] + hidden_dim + [output_dim]
        layers = []                   
        n = len(layer_dim)

        for i in range(n - 1):
            layers.append(nn.Linear(layer_dim[i], layer_dim[i + 1]))

            if i < n - 2:
                if norm == "LN":                                    
                    layers.append(nn.LayerNorm(layer_dim[i + 1]))  
                elif norm == "BN":                                 
                    layers.append(nn.BatchNorm1d(layer_dim[i + 1]))
                else:
                    raise ValueError(f"不支持的归一化: {norm}")

                if activation == "relu":
                    layers.append(nn.ReLU())
                elif activation == "tanh":
                    layers.append(nn.Tanh())
                elif activation == "sigmoid":
                    layers.append(nn.Sigmoid())
                elif activation == "gelu":                          
                    layers.append(nn.GELU())
                else:
                    raise ValueError(f"不支持的激活函数: {activation}")
                if dropout > 0:                                    
                    layers.append(nn.Dropout(dropout))

        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)


# ===== 验证 =====
model = MLP(input_dim=64, hidden_dim=[128, 128], output_dim=10, norm="LN")
x = torch.randn(32, 64)
print(model(x).shape)   

torch.Size([32, 10])


In [32]:
import torch
def cross_entropy(logits, target):
    B, D = logits.shape
    logit_max = torch.max(logits, dim=-1)[0]
    logit_stable = logits - logit_max
    logit_log_sum_exp = torch.log(torch.sum(torch.exp(logit_stable), dim=-1))
    pos_logits = logit_stable[torch.arange(B), target]
    loss = - pos_logits + logit_log_sum_exp
    return loss.mean()
logits = torch.tensor([[1.0000, 0.4985, 0.6664, 0.2533],
                    [0.4985, 1.0000, 0.8408, 0.5431],
                    [0.6664, 0.8408, 1.0000, 0.8372],
                    [0.2533, 0.5431, 0.8372, 1.0000]])
target = torch.tensor([0, 1, 2, 3])
print(cross_entropy(logits, target))

tensor(1.1176)


In [ ]:
import torch
import torch.nn.functional as F
def BCE_loss(logits, label):
    loss = torch.clamp(logits, min=0) - logits * label + torch.log(1 + torch.exp(-torch.abs(logits)))
    return loss.mean()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
def infoNce(user_emb, item_emb, temperature=1.0):
    """
    user_emb: (B, D)
    item_emb: (B, D)
    """
    user_emb = F.normalize(user_emb, dim=-1)
    item_emb = F.normalize(item_emb, dim=-1)
    sim = torch.matmul(user_emb, item_emb.transpose(0, 1)) / temperature
    label = torch.arange(user_emb.size(0))
    loss = F.cross_entropy(sim, label)
    return loss.mean()

In [ ]:
import numpy as np
from collections import defaultdict

def _auc_single_user(user_scores, user_labels):
    """单用户 AUC，处理 tie（同分算 0.5）"""
    order = np.argsort(user_scores)
    sorted_scores = user_scores[order]
    sorted_labels = user_labels[order]

    n_pos = np.sum(sorted_labels == 1)
    n_neg = np.sum(sorted_labels == 0)

    cum_neg = 0
    correct_pairs = 0.0
    l, n = 0, len(sorted_labels)

    while l < n:
        r = l
        # 找到同分的一组
        while r < n and sorted_scores[r] == sorted_scores[l]:
            r += 1
        # [l, r) 为同分组
        group_pos = np.sum(sorted_labels[l:r] == 1)
        group_neg = np.sum(sorted_labels[l:r] == 0)
        # 同分组内正负对算 0.5，组前负样本算完全正确
        correct_pairs += group_pos * cum_neg + group_pos * group_neg * 0.5
        # 注意先计算正确的配对，再更新累计负样本，因为group_neg本组内和正样本预测分数相等的
        cum_neg += group_neg
        l = r

    return correct_pairs / (n_pos * n_neg)


def gauc_rank(user_ids, labels, scores, weight_type='impression'):

    # 去掉细节，GAUC 的本质只有两步：
    # GAUC = Σ(用户i的AUC × 用户i的权重) / Σ(用户i的权重)

    user_ids = np.array(user_ids)
    labels   = np.array(labels)
    scores   = np.array(scores)

    user_sample_dict = defaultdict(list)
    for idx, uid in enumerate(user_ids):
        user_sample_dict[uid].append(idx)

    user_auc_dict      = {}
    total_weighted_auc = 0.0
    total_weight       = 0.0

    for uid, indices in user_sample_dict.items():
        user_labels = labels[indices]
        user_scores = scores[indices]

        n_pos = np.sum(user_labels == 1)
        n_neg = np.sum(user_labels == 0)

        # 无正样本或者负样本的用户，无法计算auc，直接过滤
        if n_pos == 0 or n_neg == 0:
            continue

        user_auc = _auc_single_user(user_scores, user_labels)
        user_auc_dict[uid] = user_auc

        if weight_type == 'impression':
            weight = len(indices)
        elif weight_type == 'uniform':
            weight = 1.0
        else:
            raise ValueError(f"Unknown weight_type: {weight_type}")

        total_weighted_auc += user_auc * weight
        total_weight       += weight
    
    if total_weight == 0:
        raise ValueError("No valid user found for AUC computation.")
    
    gauc = total_weighted_auc / total_weight

    return gauc, user_auc_dict


label   = [0, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0]
q       = [0.1, 0.9, 0.2, 0.8, 1, 0.2, 0.3, 0.9, 0.7, 0.9, 0.7]
user_id = [1,   1,   1,   1,   2, 2,   2,   2,   3,   3,   3  ]
gauc_rank(user_id, label, q, weight_type='impression')